# Synthetic 2D Heatmaps — Greedy / DDPath / KL-IG²

Reproduces the A_x − A_y scalar heatmap concept from
`synthetic_funcs_heatmap_attributions.py` (∇f / IG / KLIG / SHAP) and
extends it to our three methods:

- **Greedy** — `GreedyMuAttributor` (gradient-weighted μ advance)
- **DDPath** — `KLIntegratedGradients` with `DDiffusionPath` (cosine-schedule diffusion path)
- **KL-IG²** — pixel-space KL-descent IG (here adapted to the scalar setting:
  descend `−f` from the query point and integrate `∇f·Δx`)

Grid: 40×40 over `[-EXTENT, EXTENT]²`. Each column shows `A_x − A_y` for one
method; rows = the five toy functions.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1.git /content/KLIG_V1 2>/dev/null || \
    (cd /content/KLIG_V1 && git pull) 2>/dev/null
import os, sys
for _root in ['/content/KLIG_V1/infocube-main', 'infocube-main', '.']:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from __future__ import annotations
import math, time, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

from klig import KLIntegratedGradients, GreedyMuAttributor, DDiffusionPath

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
EXTENT          = 2.0
GRID_RESOLUTION = 40    # 1600 query points; bump to 60–100 for publication-quality
HEAT_N          = 200   # function heatmap resolution

# Integrator settings — kept small so all-method run finishes in minutes on CPU.
N_STEPS    = 24
N_SAMPLES  = 6
SIGMA_F    = 0.02

# KL-IG² (scalar adaptation) settings
IG2_T          = 24
IG2_STEP_SIZE  = 0.08
IG2_GRAD_CLIP  = 1.0

In [ ]:
# ── Toy 2D functions (verbatim from synthetic_funcs_heatmap_attributions.py) ─
class XOR(nn.Module):
    def __init__(self, s=5.0): super().__init__(); self.s = s
    def forward(self, x):
        return torch.tanh(self.s*x[...,0]) * torch.tanh(self.s*x[...,1])

class Checkerboard(nn.Module):
    def __init__(self, period=1.0, s=15.0):
        super().__init__(); self.period=period; self.s=s
    def forward(self, x):
        u = torch.cos(math.pi*x[...,0]/self.period)
        v = torch.cos(math.pi*x[...,1]/self.period)
        return torch.tanh(self.s * u * v)

class DiagonalCheckerboard(nn.Module):
    def __init__(self, period=1.0, s=15.0):
        super().__init__(); self.period=period; self.s=s
    def forward(self, x):
        u = (x[...,0]+x[...,1])/math.sqrt(2)
        v = (x[...,0]-x[...,1])/math.sqrt(2)
        cu = torch.cos(math.pi*u/self.period)
        cv = torch.cos(math.pi*v/self.period)
        return torch.tanh(self.s * cu * cv)

class RadialRings(nn.Module):
    def __init__(self, k=1.0, sigma=1.5):
        super().__init__(); self.k=k; self.sigma=sigma
    def forward(self, x):
        r = torch.norm(x, dim=-1)
        env = torch.exp(-r**2/(2*self.sigma**2))
        return env * torch.cos(2*math.pi*self.k*r)

class FlatFarFieldBumps(nn.Module):
    def __init__(self,
                 centers=((0.15,-0.10),(-0.12,0.18),(0.05,0.05)),
                 amps=(1.0,-0.85,0.70), sigma=0.12):
        super().__init__()
        self.register_buffer('centers', torch.tensor(centers, dtype=torch.float32))
        self.register_buffer('amps',    torch.tensor(amps,    dtype=torch.float32))
        self.sigma = sigma
    def forward(self, x):
        diff = x.unsqueeze(-2) - self.centers
        r2   = (diff**2).sum(-1)
        return (torch.exp(-r2/(2*self.sigma**2)) * self.amps).sum(-1)

FUNCS = {
    'xor':            XOR(5.0).to(DEVICE),
    'checkerboard':   Checkerboard(1.0, 15.0).to(DEVICE),
    'diagonal_ckb':   DiagonalCheckerboard(1.0, 15.0).to(DEVICE),
    'radial':         RadialRings(1.0, 1.5).to(DEVICE),
    'flat_far_field': FlatFarFieldBumps().to(DEVICE),
}
FUNC_NAMES = list(FUNCS)
print('functions:', FUNC_NAMES)

In [ ]:
# ── Grid helpers ─────────────────────────────────────────────────────────────
def grid_points(n, extent=EXTENT):
    xs = np.linspace(-extent, extent, n)
    X, Y = np.meshgrid(xs, xs, indexing='xy')
    return np.stack([X.flatten(), Y.flatten()], axis=1).astype(np.float32)

def evaluate_heat(fn, n=HEAT_N, extent=EXTENT):
    xs = torch.linspace(-extent, extent, n, device=DEVICE)
    X, Y = torch.meshgrid(xs, xs, indexing='xy')
    pts = torch.stack([X.flatten(), Y.flatten()], dim=1)
    with torch.no_grad():
        return fn(pts).cpu().numpy().reshape(n, n)

In [ ]:
# ── Reference methods: ∇f and standard IG (batched over query points) ───────
def gradient_field(fn, points):
    pts = torch.tensor(points, device=DEVICE).requires_grad_(True)
    y   = fn(pts)
    g   = torch.autograd.grad(y.sum(), pts)[0]
    return g.detach().cpu().numpy()

def integrated_gradients(fn, points, n_steps=64):
    pts   = torch.tensor(points, device=DEVICE)
    B, D  = pts.shape
    base  = torch.zeros(D, device=DEVICE)
    delta = pts - base
    attr  = torch.zeros(B, D, device=DEVICE)
    for k in range(n_steps):
        alpha = (k + 0.5) / n_steps
        xa = (base + alpha * delta).requires_grad_(True)
        g  = torch.autograd.grad(fn(xa).sum(), xa)[0]
        attr += g.detach() * delta / n_steps
    return attr.cpu().numpy()

In [ ]:
# ── Our methods: Greedy / DDPath / KL-IG² (looped over query points) ────────
# All integrate per-point — fine for a 40×40 grid; bump GRID_RESOLUTION to
# 60–100 for sharper diff maps at the cost of longer wall-time.

def _objective_callable(y):
    """Scalar objective for nn.Modules returning shape (n_samples,)."""
    return y

def greedy_field(fn, points):
    attr_ = np.zeros_like(points)
    g = GreedyMuAttributor(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                            sigma_final=SIGMA_F, device=DEVICE)
    for i in range(len(points)):
        x = torch.tensor(points[i], device=DEVICE)
        r = g.attribute(x, target=_objective_callable)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

def ddpath_field(fn, points):
    attr_ = np.zeros_like(points)
    ig = KLIntegratedGradients(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                sigma_final=SIGMA_F,
                                path=DDiffusionPath(schedule='cosine'),
                                device=DEVICE)
    for i in range(len(points)):
        x = torch.tensor(points[i], device=DEVICE)
        r = ig.attribute(x, target=_objective_callable)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

def klig2_field(fn, points,
                T=IG2_T, step_size=IG2_STEP_SIZE, grad_clip=IG2_GRAD_CLIP):
    """KL-IG²-style attribution adapted for scalar f: descend −f from x_q,
    integrate ∇f · Δx along the path.  All points evolved in parallel.
    """
    pts = torch.tensor(points, device=DEVICE)
    x_t = pts.clone()
    attr = torch.zeros_like(pts)
    for _ in range(T):
        x_in = x_t.detach().requires_grad_(True)
        y    = fn(x_in)
        g    = torch.autograd.grad(y.sum(), x_in)[0]
        # per-point grad-norm clip
        norms = g.norm(dim=-1, keepdim=True)
        scale = torch.clamp(grad_clip / (norms + 1e-12), max=1.0)
        g_c   = g * scale
        # normalised descent direction (per point)
        g_n   = g_c / (g_c.norm(dim=-1, keepdim=True) + 1e-12)
        dx    = -step_size * g_n        # descend −f → move opposite ∇f
        attr += g.detach() * dx          # ∇f · Δx (per dim)
        x_t   = (x_t + dx).detach()
    return attr.cpu().numpy()

In [ ]:
# ── Compute attributions for every function × method ────────────────────────
METHODS = [
    ('∇f',     'grad',    gradient_field),
    ('IG',     'ig',      integrated_gradients),
    ('Greedy', 'greedy',  greedy_field),
    ('DDPath', 'ddpath',  ddpath_field),
    ('KL-IG²', 'klig2',   klig2_field),
]

points = grid_points(GRID_RESOLUTION)
data = {}
for name, fn in FUNCS.items():
    print(f'[{name}]', end=' ')
    data[f'{name}_heat'] = evaluate_heat(fn)
    for mname, key, method_fn in METHODS:
        t0 = time.time()
        data[f'{name}_{key}'] = method_fn(fn, points)
        print(f'{mname}={time.time()-t0:.1f}s', end=' ')
    print()

In [ ]:
# ── Render: rows = functions, cols = f + each method's (A_x − A_y) ───────────
PERCENTILE = 100.0
MIN_REF    = 0.1
n          = GRID_RESOLUTION

nrows, ncols = len(FUNC_NAMES), 1 + len(METHODS)
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(2.6*ncols, 2.6*nrows + 0.3),
                          facecolor='white')
if nrows == 1: axes = axes.reshape(1, ncols)

for ri, name in enumerate(FUNC_NAMES):
    heat = data[f'{name}_heat']
    vmax = max(1e-6, float(np.abs(heat).max()))
    ax = axes[ri, 0]
    im = ax.imshow(heat, extent=[-EXTENT, EXTENT, -EXTENT, EXTENT],
                    origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    ax.set_ylabel(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    if ri == 0: ax.set_title('f', fontsize=11)

    for ci, (mname, key, _) in enumerate(METHODS, start=1):
        attrs = data[f'{name}_{key}']
        diff  = (attrs[:, 0] - attrs[:, 1]).reshape(n, n)
        ref   = float(np.abs(diff).max()) if PERCENTILE == 100 \
                else float(np.percentile(np.abs(diff), PERCENTILE))
        ref   = max(ref, MIN_REF)
        ax = axes[ri, ci]
        ax.imshow(diff, extent=[-EXTENT, EXTENT, -EXTENT, EXTENT],
                   origin='lower', cmap='PuOr', vmin=-ref, vmax=ref)
        if ri == 0: ax.set_title(mname, fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        ax.text(0.02, 0.97,
                 f'max={np.abs(diff).max():.2f}\nref={ref:.2f}',
                 transform=ax.transAxes, fontsize=7, va='top',
                 bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                            alpha=0.7, edgecolor='none'))

fig.suptitle(f'A_x − A_y attribution diff ({n}×{n} grid)', fontsize=12)
plt.tight_layout(rect=[0, 0.01, 1, 0.96])
plt.show()

## Reading the panels

Each non-`f` column shows `A_x − A_y` for one method. Orange = method credits
the **x-feature** more than y; purple = the opposite. The five rows expose
different failure modes:

- **xor** — saddle at origin. Methods that respect the bilinear interaction
  should show four sign-alternating quadrants. IG with a zero baseline tends
  to look bland here because the straight-line path averages the interaction
  away.
- **checkerboard / diagonal_ckb** — diagonal_ckb is only separable in the
  rotated `(x+y, x−y)` basis, so axis-aligned methods (IG, SHAP-style) will
  smear blame across both features symmetrically; KL-path / greedy methods
  do somewhat better.
- **radial** — by symmetry `A_x − A_y` should be zero on the diagonals
  and antisymmetric across them. Any method that violates this on average
  is leaking baseline geometry into the attribution.
- **flat_far_field** — far from the bumps the function is flat, so the
  ground-truth A_x − A_y is ≈0 there. Any non-trivial colour in the far
  field is the IG "shadow effect" (a straight path through the bump
  cluster). KL-IG² and DDPath should be visibly cleaner there.

To sharpen the maps, bump `GRID_RESOLUTION` to 60–100 (longer wall-time)
and/or increase `N_STEPS`, `N_SAMPLES`.